# 03 — Analysis
**Project:** HSR Decay in the UPL  
**Purpose:** Statistical testing of HSR decay — existence, positional variation, outcome variation  
**Input:** `data/processed/upl_hsr_clean.csv`  
**Output:** Summary tables to `outputs/tables/`

In [26]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

CLEAN_FILE = '../data/processed/upl_hsr_clean.csv'

df = pd.read_csv(CLEAN_FILE)
print(f'Rows loaded: {len(df):,}')
print(f'Unique player-match pairs: {df["player_match_id"].nunique():,}')

Rows loaded: 9,990
Unique player-match pairs: 4,992


## 1. Pivot to Wide Format
Each row = one player-match observation with H1 and H2 side by side

In [27]:
wide = df.pivot_table(
    index=['player_match_id', 'player_name', 'club_for', 'match_day',
           'session_title', 'position', 'result'],
    columns='split_name',
    values='hsr_rate'
).reset_index()

wide.columns.name = None
wide = wide.rename(columns={'1st Half': 'h1_rate', '2nd Half': 'h2_rate'})
wide = wide.dropna(subset=['h1_rate', 'h2_rate']) #only players with valid data for both halves retained(removes substituted ,injured or incomplete player data)

wide = wide[wide['h1_rate'] >=1] #Players with H1 HSR rates below 1 m/min contributed negligible high-speed output in the first half and were excluded, as decay from a near-zero baseline is not analytically meaningful

# Decay variables
wide['decay_abs'] = wide['h1_rate'] - wide['h2_rate']          # absolute drop (m/min)

#A positive value indicates a decline in running intensity (physical fatigue or tactical adjustment), while a negative value suggests increased intensity in the second half (tactical shift, momentum change, or opponent pressure)
print(f'Valid paired observations: {len(wide):,}')
print(wide[['h1_rate', 'h2_rate', 'decay_abs' ]].describe().round(2))

Valid paired observations: 3,441
       h1_rate  h2_rate  decay_abs
count  3441.00  3441.00    3441.00
mean     10.80     8.11       2.69
std       4.22     4.22       4.37
min       1.27     0.00     -22.61
25%       7.59     5.29      -0.09
50%      10.47     7.70       2.05
75%      13.62    10.70       4.86
max      31.74    26.38      26.30


## 2. Normality Check
Paired t-test assumes the differences are approximately normally distributed

In [28]:
stat, p_shapiro = stats.shapiro(wide['decay_abs'].sample(min(5000, len(wide)), random_state=42))
print(f'Shapiro-Wilk on decay_abs: W={stat:.4f}, p={p_shapiro:.4f}')

if p_shapiro < 0.05:
    print('Distribution is non-normal (p < 0.05).')
    print('Will run BOTH paired t-test and Wilcoxon signed-rank test.')
    print('With n > 1000, t-test is robust to non-normality (Central Limit Theorem).')
    print('Report both; if they agree, conclusion stands.')
else:
    print('Distribution is approximately normal. Paired t-test is appropriate.')

Shapiro-Wilk on decay_abs: W=0.9567, p=0.0000
Distribution is non-normal (p < 0.05).
Will run BOTH paired t-test and Wilcoxon signed-rank test.
With n > 1000, t-test is robust to non-normality (Central Limit Theorem).
Report both; if they agree, conclusion stands.


## 3. Does HSR Decay Exist? — Paired T-Test + Wilcoxon

In [29]:
t_stat, p_ttest = stats.ttest_rel(wide['h1_rate'], wide['h2_rate'])
w_stat, p_wilcox = stats.wilcoxon(wide['h1_rate'], wide['h2_rate'])

# Effect size: Cohen's d for paired data
d = wide['decay_abs'].mean() / wide['decay_abs'].std()

print('=' * 55)
print('PART 1: DOES HSR DECAY EXIST?')
print('=' * 55)
print(f'Mean H1 HSR Rate:        {wide["h1_rate"].mean():.2f} m/min')
print(f'Mean H2 HSR Rate:        {wide["h2_rate"].mean():.2f} m/min')
print(f'Mean absolute decay:     {wide["decay_abs"].mean():.2f} m/min')
print()
print(f'Paired t-test:  t={t_stat:.3f}, p={p_ttest:.6f}')
print(f'Wilcoxon:       W={w_stat:.1f}, p={p_wilcox:.6f}')
print(f'Effect size:    Cohen\'s d = {d:.3f}')
print()
if p_ttest < 0.05:
    print('CONCLUSION: HSR decay is statistically significant (p < 0.05).')
    if abs(d) < 0.2:
        print('Effect size: negligible')
    elif abs(d) < 0.5:
        print('Effect size: small')
    elif abs(d) < 0.8:
        print('Effect size: medium')
    else:
        print('Effect size: large')
else:
    print('CONCLUSION: No statistically significant HSR decay detected (p >= 0.05).')
print('=' * 55)

# Save
part1 = pd.DataFrame([{
    'mean_h1': round(wide['h1_rate'].mean(), 2),
    'mean_h2': round(wide['h2_rate'].mean(), 2),
    'mean_decay_abs': round(wide['decay_abs'].mean(), 2),
    't_stat': round(t_stat, 3),
    'p_ttest': round(p_ttest, 4),
    'cohens_d': round(d, 3)
}])
part1.to_csv('../output/tables/part1_overall_decay.csv', index=False)
print('Saved: output/tables/part1_overall_decay.csv')

PART 1: DOES HSR DECAY EXIST?
Mean H1 HSR Rate:        10.80 m/min
Mean H2 HSR Rate:        8.11 m/min
Mean absolute decay:     2.69 m/min

Paired t-test:  t=36.055, p=0.000000
Wilcoxon:       W=1038115.0, p=0.000000
Effect size:    Cohen's d = 0.615

CONCLUSION: HSR decay is statistically significant (p < 0.05).
Effect size: medium
Saved: output/tables/part1_overall_decay.csv


## 4. Does Decay Vary by Position? — One-Way ANOVA + Tukey HSD

In [30]:
positions = ['DEF', 'MID', 'FWD']
groups_pos = [wide[wide['position'] == p]['decay_abs'].dropna() for p in positions]

f_stat, p_anova_pos = stats.f_oneway(*groups_pos)

print('=' * 55)
print('PART 2: DOES DECAY VARY BY POSITION?')
print('=' * 55)
print(f'One-way ANOVA: F={f_stat:.3f}, p={p_anova_pos:.4f}')
print()

# Descriptives per position
pos_summary = wide.groupby('position')['decay_abs'].agg(
    n='count',
    mean_decay=lambda x: round(x.mean(), 2),
    sd=lambda x: round(x.std(), 2),
    pct_decay=lambda x: round((x / wide.loc[x.index, 'h1_rate']).mean() * 100, 1)
).loc[positions]
print(pos_summary)
print()

if p_anova_pos < 0.05:
    print('Significant positional difference detected. Running Tukey HSD...')
    print()
    tukey_pos = pairwise_tukeyhsd(
        endog=wide['decay_abs'],
        groups=wide['position'],
        alpha=0.05
    )
    print(tukey_pos.summary())
else:
    print('No significant positional difference (p >= 0.05). Tukey HSD not required.')
print('=' * 55)

pos_summary.to_csv('../output/tables/part2_decay_by_position.csv')
print('Saved: output/tables/part2_decay_by_position.csv')

PART 2: DOES DECAY VARY BY POSITION?
One-way ANOVA: F=40.253, p=0.0000

             n  mean_decay    sd  pct_decay
position                                   
DEF       1491        1.98  4.12       12.8
MID       1122        2.95  4.16       22.9
FWD        828        3.60  4.86       23.9

Significant positional difference detected. Running Tukey HSD...

Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
   DEF    FWD   1.6175    0.0  1.1781  2.0568   True
   DEF    MID   0.9623    0.0  0.5617  1.3629   True
   FWD    MID  -0.6552 0.0027 -1.1196 -0.1907   True
----------------------------------------------------
Saved: output/tables/part2_decay_by_position.csv


## 5. Does Decay Vary by Match Outcome? — One-Way ANOVA + Tukey HSD

In [31]:
outcomes = ['Win', 'Draw', 'Loss']
groups_out = [wide[wide['result'] == o]['decay_abs'].dropna() for o in outcomes]

f_stat_o, p_anova_out = stats.f_oneway(*groups_out)

print('=' * 55)
print('PART 3: DOES DECAY VARY BY MATCH OUTCOME?')
print('=' * 55)
print(f'One-way ANOVA: F={f_stat_o:.3f}, p={p_anova_out:.4f}')
print()

out_summary = wide.groupby('result')['decay_abs'].agg(
    n='count',
    mean_decay=lambda x: round(x.mean(), 2),
    sd=lambda x: round(x.std(), 2)
).loc[outcomes]
print(out_summary)
print()

if p_anova_out < 0.05:
    print('Significant outcome difference detected. Running Tukey HSD...')
    print()
    tukey_out = pairwise_tukeyhsd(
        endog=wide['decay_abs'],
        groups=wide['result'],
        alpha=0.05
    )
    print(tukey_out.summary())
else:
    print('No significant outcome difference (p >= 0.05). Tukey HSD not required.')
print('=' * 55)

out_summary.to_csv('../output/tables/part3_decay_by_outcome.csv')
print('Saved: output/tables/part3_decay_by_outcome.csv')

PART 3: DOES DECAY VARY BY MATCH OUTCOME?
One-way ANOVA: F=1.121, p=0.3260

           n  mean_decay    sd
result                        
Win     1356        2.60  4.29
Draw     966        2.62  3.84
Loss    1119        2.85  4.87

No significant outcome difference (p >= 0.05). Tukey HSD not required.
Saved: output/tables/part3_decay_by_outcome.csv


In [32]:
club_results = []

for club, group in wide.groupby('club_for'):
    if len(group) < 10:  # skip clubs with too few observations
        continue
    t, p = stats.ttest_rel(group['h1_rate'], group['h2_rate'])
    club_results.append({
        'club': club,
        'n': len(group),
        'mean_h1': round(group['h1_rate'].mean(), 2),
        'mean_h2': round(group['h2_rate'].mean(), 2),
        'mean_decay': round(group['decay_abs'].mean(), 2),
        't_stat': round(t, 3),
        'p_value': round(p, 4),
        'significant': p < 0.05
    })

club_df = pd.DataFrame(club_results).sort_values('mean_decay', ascending=False)
print(club_df.to_string(index=False))
club_df.to_csv('../output/tables/part1b_decay_by_club.csv', index=False)

   club   n  mean_h1  mean_h2  mean_decay  t_stat  p_value  significant
Club-04 257    12.42     8.08        4.34  14.234      0.0         True
Club-06 278    10.56     6.86        3.70  12.322      0.0         True
Club-03  76    11.25     7.68        3.57   7.297      0.0         True
Club-05 266    10.33     7.15        3.18  12.119      0.0         True
Club-16 224    10.58     7.53        3.05  10.041      0.0         True
Club-14 257    11.26     8.35        2.91   9.265      0.0         True
Club-09 270    10.64     7.83        2.81   9.272      0.0         True
Club-15 261    11.24     8.44        2.80  11.704      0.0         True
Club-07 237    12.19     9.67        2.52   8.088      0.0         True
Club-08  77    11.38     9.02        2.37   5.042      0.0         True
Club-02 247     9.06     6.83        2.22   9.132      0.0         True
Club-12 132    12.72    10.53        2.19   6.813      0.0         True
Club-11 171    10.16     8.20        1.96   7.927      0.0      

## 6. Save Wide Dataset for Visualisation Notebook

In [33]:
wide.to_csv('../data/processed/upl_hsr_wide.csv', index=False)
print(f'Saved: data/processed/upl_hsr_wide.csv')
print(f'Shape: {wide.shape}')

Saved: data/processed/upl_hsr_wide.csv
Shape: (3441, 10)
